# 05. 환각 탐지 & 응답 품질 평가 🏪

## 학습 목표
- LLM Hallucination의 유형과 탐지 방법 이해
- Self-consistency, Cross-model verification 실습
- Factual Grounding 기법 학습
- ai-ipsonum 매장 존재 여부 검증 파이프라인 구축
- unmatched_mentions 분석
- 평가 대시보드 시각화

## ai-ipsonum 연계 🏪
- AI가 추천한 매장이 실제 존재하는지 확인
- `unmatched_mentions` 데이터 분석: 실제 존재 vs 순수 hallucination

---

In [ ]:
import json
import re
import random
from collections import Counter
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import numpy as np
import pandas as pd

## 1. LLM Hallucination 유형

Hallucination은 LLM이 **사실이 아니거나 근거 없는 정보를 자신 있게 생성**하는 현상.

### 3가지 주요 유형

| 유형 | 설명 | 예시 |
|------|------|------|
| **Factual Hallucination** | 사실과 다른 정보 생성 | "블루보틀은 2015년 설립" (실제 2017년) |
| **Faithfulness Hallucination** | 주어진 컨텍스트와 다른 답변 | RAG에서 문서에 없는 내용 추가 |
| **Instruction Violation** | 지시를 따르지 않음 | JSON 요청에 텍스트로 응답 |

### ai-ipsonum에서의 Hallucination

- AI가 "매머드 카페"를 추천 → 실제 존재하지 않는 매장
- AI가 "블루보틀 강남점"을 추천 → 실제 없는 지점명

In [ ]:
# --- Hallucination 예시 데이터 ---

# AI가 추출한 매장 리스트 (여러 쿼리 결과)
ai_recommendations = [
    {
        "query": "성수동 카페 TOP 5",
        "engine": "ChatGPT",
        "stores": [
            {"name": "블루보틀 성수점", "rank": 1},
            {"name": "스타벅스 종로점", "rank": 2},
            {"name": "컨테이너 성수동", "rank": 3},
            {"name": "매머드 카페", "rank": 4},       # hallucination
            {"name": "솔길체", "rank": 5},
        ]
    },
    {
        "query": "성수동 카페 TOP 5",
        "engine": "Perplexity",
        "stores": [
            {"name": "블루보틀 성수점", "rank": 1},
            {"name": "컨테이너 성수동", "rank": 2},
            {"name": "플릿화이트", "rank": 3},
            {"name": "솔길체", "rank": 4},
            {"name": "리보 커피", "rank": 5},           # hallucination
        ]
    },
    {
        "query": "강남 맛집 TOP 5",
        "engine": "ChatGPT",
        "stores": [
            {"name": "유육의 달인 강남점", "rank": 1},  # 실제 존재하지만 DB에 없음
            {"name": "오마카세 강남점", "rank": 2},
            {"name": "딱타이 강남", "rank": 3},
            {"name": "봉펼 하우스", "rank": 4},         # hallucination
            {"name": "아움 정식", "rank": 5},           # 실제 존재하지만 DB에 없음
        ]
    }
]

# 실제 매장 DB (Ground Truth)
store_database = {
    "블루보틀 성수점": {"exists": True, "address": "성수동 123", "verified": True},
    "스타벅스 종로점": {"exists": True, "address": "종로 456", "verified": True},
    "컨테이너 성수동": {"exists": True, "address": "성수동 789", "verified": True},
    "솔길체": {"exists": True, "address": "성수동 101", "verified": True},
    "플릿화이트": {"exists": True, "address": "성수동 102", "verified": True},
    "오마카세 강남점": {"exists": True, "address": "강남 200", "verified": True},
    "딱타이 강남": {"exists": True, "address": "강남 201", "verified": True},
}

print(f"AI 추천 쿼리 수: {len(ai_recommendations)}")
print(f"DB 등록 매장 수: {len(store_database)}")

---
## 2. Hallucination 탐지 방법

### 2.1 Self-Consistency

동일한 질문을 여러 번 호출하여 **일관성 없는 답변**을 탐지.
- 일관되지 않은 답변 = hallucination 가능성 높음
- 모든 응답에 동일하게 등장하는 매장 = 신뢰도 높음

In [ ]:
# Self-Consistency 시뮬레이션

def self_consistency_check(responses: list[list[str]], threshold: float = 0.5) -> dict:
    """
    여러 응답에서 일관성을 확인.
    threshold 이상 반복된 매장 = consistent
    """
    all_mentions = Counter()
    n_responses = len(responses)
    
    for resp in responses:
        for store in set(resp):  # 응답 내 중복 제거
            all_mentions[store] += 1
    
    consistent = {}
    inconsistent = {}
    
    for store, count in all_mentions.items():
        ratio = count / n_responses
        if ratio >= threshold:
            consistent[store] = {"count": count, "ratio": ratio}
        else:
            inconsistent[store] = {"count": count, "ratio": ratio}
    
    return {"consistent": consistent, "inconsistent": inconsistent}


# 5회 응답 시뮬레이션
simulated_responses = [
    ["블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동", "솔길체"],
    ["블루보틀 성수점", "컨테이너 성수동", "플릿화이트", "매머드 카페"],
    ["블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동", "솔길체"],
    ["블루보틀 성수점", "컨테이너 성수동", "플릿화이트", "리보 커피"],
    ["블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동", "솔길체", "플릿화이트"],
]

result = self_consistency_check(simulated_responses, threshold=0.5)

print("=== Self-Consistency 결과 (5회 응답) ===")
print("\n\u2713 Consistent (신뢰 높음):")
for store, info in sorted(result["consistent"].items(), key=lambda x: -x[1]["ratio"]):
    print(f"  {store}: {info['count']}/5 ({info['ratio']:.0%})")

print("\n\u2717 Inconsistent (hallucination 의심):")
for store, info in result["inconsistent"].items():
    print(f"  {store}: {info['count']}/5 ({info['ratio']:.0%})")

### 2.2 Cross-Model Verification

여러 모델의 응답을 교차 검증하여 hallucination 탐지.

In [ ]:
# Cross-Model Verification

def cross_model_verify(recommendations: list[dict]) -> dict:
    """여러 엔진 결과를 교차 검증"""
    # 동일 쿼리에 대한 결과만 그룹화
    by_query = {}
    for rec in recommendations:
        query = rec["query"]
        if query not in by_query:
            by_query[query] = {}
        by_query[query][rec["engine"]] = [s["name"] for s in rec["stores"]]
    
    verification_results = {}
    
    for query, engine_results in by_query.items():
        store_votes = Counter()
        for stores in engine_results.values():
            for store in stores:
                store_votes[store] += 1
        
        n_engines = len(engine_results)
        verified = {s: v for s, v in store_votes.items() if v > 1}  # 2개 이상 엔진에서 언급
        unverified = {s: v for s, v in store_votes.items() if v == 1}
        
        verification_results[query] = {
            "verified": verified,
            "unverified": unverified,
            "n_engines": n_engines
        }
    
    return verification_results


# 성수동 쿼리에 대한 2개 엔진 결과 교차 검증
verification = cross_model_verify(ai_recommendations[:2])  # 성수동 쿼리만

print("=== Cross-Model Verification ===")
for query, result in verification.items():
    print(f"\nQuery: \"{query}\" ({result['n_engines']} engines)")
    print("  \u2713 Verified (2+ engines):")
    for store, votes in sorted(result["verified"].items(), key=lambda x: -x[1]):
        print(f"    {store}: {votes} engines")
    print("  \u2717 Unverified (1 engine only):")
    for store, votes in result["unverified"].items():
        print(f"    {store}: {votes} engine \u2192 hallucination \uc758\uc2ec")

---
## 3. Factual Grounding: 외부 소스로 사실 확인

AI의 응답을 **외부 데이터베이스**와 대조하여 사실 여부를 확인.

In [ ]:
# Factual Grounding 구현

def verify_store_existence(store_name: str, database: dict) -> dict:
    """
    매장이 DB에 존재하는지 확인.
    실제로는 네이버 지도 API / Google Places API 활용.
    """
    # 정확한 매칭
    if store_name in database:
        return {
            "status": "verified",
            "match_type": "exact",
            "store_name": store_name,
            "db_info": database[store_name]
        }
    
    # 부분 매칭 (퍼지 매칭 시뮬레이션)
    for db_name in database:
        if store_name in db_name or db_name in store_name:
            return {
                "status": "partial_match",
                "match_type": "fuzzy",
                "store_name": store_name,
                "matched_to": db_name,
                "db_info": database[db_name]
            }
    
    # 매칭 실패
    return {
        "status": "unverified",
        "match_type": "none",
        "store_name": store_name,
        "note": "DB에서 찾을 수 없음 - hallucination 또는 DB 미등록"
    }


# 모든 추천 매장 검증
print("=== Factual Grounding 결과 ===")
all_verification_results = []

for rec in ai_recommendations:
    print(f"\nQuery: \"{rec['query']}\" (Engine: {rec['engine']})")
    for store in rec["stores"]:
        result = verify_store_existence(store["name"], store_database)
        all_verification_results.append({
            **result,
            "query": rec["query"],
            "engine": rec["engine"],
            "rank": store["rank"]
        })
        
        status_icon = {"verified": "\u2713", "partial_match": "~", "unverified": "\u2717"}[result["status"]]
        print(f"  {status_icon} #{store['rank']} {store['name']} [{result['status']}]")

---
## 4. 🏪 매장 존재 여부 검증 파이프라인

### 실제 검증 흐름 (Mock 데이터 사용)

```
AI 추천 매장
    │
    ├─ 1차: 내부 DB 검색
    │    ├─ \uc815\ud655 \ub9e4\uce6d \u2192 \u2713 verified
    │    ├─ \ud37c\uc9c0 \ub9e4\uce6d \u2192 ~ partial_match
    │    └─ \uc5c6\uc74c \u2192 2\ucc28 \uac80\uc0c9
    │
    ├─ 2차: 외부 API 검색 (네이버/Google Mock)
    │    ├─ \ubc1c\uacac \u2192 "\uc2e4\uc81c \uc874\uc7ac\ud558\uc9c0\ub9cc DB\uc5d0 \uc5c6\uc74c"
    │    └─ \ubbf8\ubc1c\uacac \u2192 "\uc21c\uc218 hallucination"
    │
    └─ 결과 분류
```

In [ ]:
# --- Mock 외부 API (네이버 지도 / Google Places) ---

def mock_naver_map_api(store_name: str) -> Optional[dict]:
    """네이버 지도 API 시뮬레이션 (Mock)"""
    # 실제로 존재하지만 DB에는 없는 매장들
    naver_data = {
        "유육의 달인 강남점": {"name": "유육의달인 강남점", "category": "맛집", "rating": 4.5, "review_count": 1200},
        "아움 정식": {"name": "아움 정식", "category": "맛집", "rating": 4.3, "review_count": 800},
        # 이하는 네이버에도 없음 -> 순수 hallucination
    }
    return naver_data.get(store_name)


def full_verification_pipeline(store_name: str, internal_db: dict) -> dict:
    """전체 검증 파이프라인"""
    result = {
        "store_name": store_name,
        "internal_db": False,
        "external_api": False,
        "final_status": "unknown",
        "classification": "unknown"
    }
    
    # 1차: 내부 DB 검색
    if store_name in internal_db:
        result["internal_db"] = True
        result["final_status"] = "verified"
        result["classification"] = "confirmed"
        return result
    
    # 2차: 외부 API 검색
    external = mock_naver_map_api(store_name)
    if external:
        result["external_api"] = True
        result["final_status"] = "exists_not_in_db"
        result["classification"] = "db_gap"  # DB에 없을 뿐, 실제 존재
        result["external_info"] = external
        return result
    
    # 둘 다 없음
    result["final_status"] = "not_found"
    result["classification"] = "hallucination"  # 순수 hallucination
    return result


# 전체 검증 실행
print("=== 전체 검증 파이프라인 결과 ===")
pipeline_results = []

for rec in ai_recommendations:
    for store in rec["stores"]:
        result = full_verification_pipeline(store["name"], store_database)
        result["query"] = rec["query"]
        result["engine"] = rec["engine"]
        result["rank"] = store["rank"]
        pipeline_results.append(result)
        
        icon = {"confirmed": "\u2713", "db_gap": "\u25b3", "hallucination": "\u2717"}[result["classification"]]
        print(f"  {icon} {store['name']} [{result['classification']}] "
              f"(internal_db={result['internal_db']}, external={result['external_api']})")

---
## 5. 🏪 unmatched_mentions 분석

ai-ipsonum에서 `unmatched_mentions`는 **AI가 언급했지만 DB에서 매칭되지 않은 매장명**을 말한다.

### 분류 기준

| 분류 | 설명 | 조치 |
|------|------|------|
| **db_gap** | 실제 존재하지만 DB에 없음 | DB에 추가 등록 |
| **hallucination** | 실제 존재하지 않음 | 필터링 필요 |
| **name_variation** | 이름이 조금 다름 | 매칭 로직 개선 |

In [ ]:
# --- unmatched_mentions 샘플 데이터 ---

unmatched_mentions = [
    {"mention": "매머드 카페", "query": "성수동 카페", "engine": "ChatGPT", "count": 3},
    {"mention": "리보 커피", "query": "성수동 카페", "engine": "Perplexity", "count": 2},
    {"mention": "유육의 달인 강남점", "query": "강남 맛집", "engine": "ChatGPT", "count": 5},
    {"mention": "아움 정식", "query": "강남 맛집", "engine": "ChatGPT", "count": 4},
    {"mention": "봉폼 하우스", "query": "강남 맛집", "engine": "ChatGPT", "count": 1},
    {"mention": "블루보틀", "query": "성수동 카페", "engine": "Gemini", "count": 2},    # 이름 변형
    {"mention": "스벅 종로", "query": "성수동 카페", "engine": "Gemini", "count": 1},   # 이름 변형
    {"mention": "드림 카페", "query": "성수동 카페", "engine": "Gemini", "count": 1},    # hallucination
]


def classify_unmatched(mention: str, internal_db: dict) -> str:
    """
    unmatched mention을 분류.
    """
    # 1. 외부 API로 실제 존재 확인
    external = mock_naver_map_api(mention)
    if external:
        return "db_gap"  # 실제 존재, DB에 없을 뿐
    
    # 2. 이름 변형 확인 (퍼지 매칭)
    for db_name in internal_db:
        # 간단한 퍼지 매칭
        if mention in db_name or db_name.startswith(mention) or any(
            word in db_name for word in mention.split() if len(word) >= 2
        ):
            return "name_variation"
    
    # 3. 순수 hallucination
    return "hallucination"


# unmatched_mentions 분류
print("=== unmatched_mentions \ubd84\ub958 ===")
classifications = []

for mention_data in unmatched_mentions:
    cls = classify_unmatched(mention_data["mention"], store_database)
    classifications.append(cls)
    
    icon = {"db_gap": "\u25b3", "name_variation": "~", "hallucination": "\u2717"}[cls]
    print(f"  {icon} \"{mention_data['mention']}\" [{cls}] "
          f"(\uc5d4\uc9c4: {mention_data['engine']}, \ud69f\uc218: {mention_data['count']})")

# 분류 통계
cls_counter = Counter(classifications)
print(f"\n\ubd84\ub958 \ud1b5\uacc4:")
for cls, count in cls_counter.most_common():
    print(f"  {cls}: {count}건 ({count/len(classifications):.0%})")

In [ ]:
# unmatched_mentions 분류 시각화

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 분류 분포
ax = axes[0]
labels = list(cls_counter.keys())
sizes = list(cls_counter.values())
colors = {"hallucination": "#ff6b6b", "db_gap": "#f7dc6f", "name_variation": "#4ecdc4"}
pie_colors = [colors.get(l, "gray") for l in labels]

ax.pie(sizes, labels=labels, colors=pie_colors, autopct='%1.0f%%',
       startangle=90, textprops={'fontsize': 11})
ax.set_title("Unmatched Mentions Classification", fontsize=12, fontweight='bold')

# 오른쪽: 엔진별 hallucination 비율
ax = axes[1]
engine_stats = {}
for mention_data, cls in zip(unmatched_mentions, classifications):
    engine = mention_data["engine"]
    if engine not in engine_stats:
        engine_stats[engine] = {"total": 0, "hallucination": 0}
    engine_stats[engine]["total"] += 1
    if cls == "hallucination":
        engine_stats[engine]["hallucination"] += 1

engine_names = list(engine_stats.keys())
hall_rates = [engine_stats[e]["hallucination"] / engine_stats[e]["total"] 
              for e in engine_names]

bars = ax.bar(engine_names, hall_rates, color=["#ff6b6b", "#45b7d1", "#4ecdc4"],
              edgecolor='black', linewidth=0.5)
ax.set_ylabel("Hallucination Rate")
ax.set_title("Hallucination Rate by Engine", fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

for bar, rate in zip(bars, hall_rates):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{rate:.0%}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 6. 평가 대시보드: 정확도, hallucination 비율 시각화

In [ ]:
# --- 전체 평가 대시보드 ---

# 파이프라인 결과에서 통계 생성
df_results = pd.DataFrame(pipeline_results)

# 전체 통계
total = len(df_results)
confirmed = len(df_results[df_results["classification"] == "confirmed"])
db_gap = len(df_results[df_results["classification"] == "db_gap"])
hallucination = len(df_results[df_results["classification"] == "hallucination"])

print("=== 평가 대시보드 ===")
print(f"\n전체 추천 매장 수: {total}")
print(f"  \u2713 확인 (confirmed): {confirmed} ({confirmed/total:.0%})")
print(f"  \u25b3 DB 미등록 (db_gap): {db_gap} ({db_gap/total:.0%})")
print(f"  \u2717 Hallucination: {hallucination} ({hallucination/total:.0%})")
print(f"\n정확도 (Precision): {confirmed / total:.1%}")
print(f"Hallucination \ube44\uc728: {hallucination / total:.1%}")

# 엔진별 정확도
print("\n\uc5d4\uc9c4\ubcc4 \uc815\ud655\ub3c4:")
for engine in df_results["engine"].unique():
    engine_data = df_results[df_results["engine"] == engine]
    eng_confirmed = len(engine_data[engine_data["classification"] == "confirmed"])
    eng_total = len(engine_data)
    eng_hall = len(engine_data[engine_data["classification"] == "hallucination"])
    print(f"  {engine}: accuracy={eng_confirmed/eng_total:.0%}, "
          f"hallucination={eng_hall/eng_total:.0%}")

In [ ]:
# 대시보드 시각화

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 전체 분류 분포
ax = axes[0, 0]
cls_counts = df_results["classification"].value_counts()
colors_map = {"confirmed": "#4ecdc4", "db_gap": "#f7dc6f", "hallucination": "#ff6b6b"}
pie_colors = [colors_map.get(c, "gray") for c in cls_counts.index]
ax.pie(cls_counts.values, labels=cls_counts.index, colors=pie_colors,
       autopct='%1.0f%%', startangle=90, textprops={'fontsize': 11})
ax.set_title("Overall Classification", fontsize=12, fontweight='bold')

# 2. 엔진별 정확도
ax = axes[0, 1]
engine_accuracy = []
engine_names = []
for engine in df_results["engine"].unique():
    engine_data = df_results[df_results["engine"] == engine]
    acc = len(engine_data[engine_data["classification"] == "confirmed"]) / len(engine_data)
    engine_accuracy.append(acc)
    engine_names.append(engine)

bars = ax.bar(engine_names, engine_accuracy, color=["#4ecdc4", "#45b7d1", "#f7dc6f"],
              edgecolor='black', linewidth=0.5)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy by Engine", fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, engine_accuracy):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{acc:.0%}', ha='center', va='bottom', fontweight='bold')

# 3. 순위별 hallucination 비율
ax = axes[1, 0]
rank_hall = []
for rank in sorted(df_results["rank"].unique()):
    rank_data = df_results[df_results["rank"] == rank]
    hall_rate = len(rank_data[rank_data["classification"] == "hallucination"]) / len(rank_data)
    rank_hall.append(hall_rate)

ranks = sorted(df_results["rank"].unique())
ax.bar(ranks, rank_hall, color=["#4ecdc4" if r < 0.3 else "#ff6b6b" for r in rank_hall],
       edgecolor='black', linewidth=0.5)
ax.set_xlabel("Rank")
ax.set_ylabel("Hallucination Rate")
ax.set_title("Hallucination Rate by Rank", fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

# 4. 쿼리별 정확도
ax = axes[1, 1]
query_stats = []
for query in df_results["query"].unique():
    q_data = df_results[df_results["query"] == query]
    q_confirmed = len(q_data[q_data["classification"] == "confirmed"])
    q_hall = len(q_data[q_data["classification"] == "hallucination"])
    q_total = len(q_data)
    query_stats.append({
        "query": query[:15],
        "accuracy": q_confirmed / q_total,
        "hallucination": q_hall / q_total,
    })

q_df = pd.DataFrame(query_stats)
x = np.arange(len(q_df))
width = 0.35
ax.bar(x - width/2, q_df["accuracy"], width, label="Accuracy", color="#4ecdc4")
ax.bar(x + width/2, q_df["hallucination"], width, label="Hallucination", color="#ff6b6b")
ax.set_xlabel("Query")
ax.set_ylabel("Rate")
ax.set_title("Accuracy vs Hallucination by Query", fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(q_df["query"], fontsize=9)
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

plt.suptitle("Hallucination Detection Dashboard", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 개선된 퍼지 매칭 구현

`unmatched_mentions`에서 `name_variation`을 더 정확하게 탐지하는 함수를 구현하세요.  
요구사항:
- "블루보틀" → "블루보틀 성수점" 매칭
- "스벅 종로" → "스타벅스 종로점" 매칭
- 레벌슈타인 거리(edit distance) 활용
- 임계값 설정 (유사도 0.6 이상)

In [ ]:
# TODO: 퍼지 매칭 함수 구현
# 요구사항:
# 1. 레벌슈타인 거리 구현 (\ub610\ub294 difflib.SequenceMatcher \uc0ac\uc6a9)
# 2. \uc784\uacc4\uac12 \uc774\uc0c1\uc774\uba74 name_variation\uc73c\ub85c \ubd84\ub958
# 3. \uc704 unmatched_mentions \ub370\uc774\ud130\ub85c \ud14c\uc2a4\ud2b8

# def fuzzy_match(mention: str, database: dict, threshold: float = 0.6) -> Optional[str]:
#     TODO
#     return None


---
## 핵심 정리

| 개념 | 설명 | ai-ipsonum 적용 |
|------|------|-------------|
| Factual Hallucination | 사실과 다른 정보 | 없는 매장 추천 |
| Self-Consistency | 여러 번 호출하여 일관성 확인 | 매장 신뢰도 측정 |
| Cross-Model Verification | 여러 모델 교차 검증 | 3-engine 검증 |
| Factual Grounding | 외부 소스로 사실 확인 | 네이버/Google API |
| unmatched_mentions | 매칭 실패한 언급 분석 | DB 개선, 필터링 |
| 평가 대시보드 | 정확도/hallucination 시각화 | 운영 모니터링 |

**다음 노트북**: [06-agents-and-tools.ipynb](06-agents-and-tools.ipynb) - Agents & Tools